In [1]:
# Importation des librairies

from __future__ import print_function
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn import metrics
from sklearn import tree
import warnings
warnings.filterwarnings('ignore')

In [4]:
df1 = pd.read_csv('datas/First_Crop_recommendation.csv')
df2 = pd.read_csv('datas/Second_Crop_recommendation.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'datas/First_Crop_recommendation.csv'

In [ ]:
print("Dataset 1: Premieres lignes")
print(df1.head())

print("\nDataset 2: Premieres lignes")
print(df2.head())

In [ ]:
print("Dataset 1: Dernieres lignes")
print(df1.tail())

print("\nDataset 2: Dernieres lignes")
print(df2.tail())

In [ ]:
print("Dataset 1: Taille = ", df1.size)
print("Dataset 2: Taille = ", df2.size)

In [ ]:
print("Dataset 1: Formes = ", df1.shape)
print("Dataset 2: Formes = ", df2.shape)

In [ ]:
print("Dataset 1: Colonnes: \n", df1.columns)
print("\nDataset 2: Colonnes: \n", df2.columns)

In [ ]:
print("Dataset 1: Labels = ", df1['label'].unique())
print("\nDataset 2: Labels = ", df2['label'].unique())

In [ ]:
print("Dataset 1: Types: \n", df1.dtypes)
print("\nDataset 2: Types: \n", df2.dtypes)

In [ ]:
# Checking Missing Values
print("Dataset 1: Valeurs Manquantes:")
print(df1.isnull().sum())

print("\nDataset 2: Valeurs Manquantes:")
print(df2.isnull().sum())

In [ ]:
# Fusion des Datasets 

df = pd.concat([df1, df2])

df = df.drop_duplicates()
df.info()

In [ ]:
df.size

In [ ]:
df

In [ ]:
df['label'].unique()

In [ ]:
df['label'].value_counts()

In [ ]:
# over all distribution 

plt.rcParams['figure.figsize'] = (10, 10) 
plt.rcParams['figure.dpi'] = 60

features = ['N', 'P', 'K', 'temperature', 
			'humidity', 'ph', 'rainfall'] 

for i, feat in enumerate(features): 
	plt.subplot(4, 2, i + 1) 
	sns.distplot(df[feat], color='greenyellow') 
	if i < 3: 
		plt.title(f'Ratio of {feat}', fontsize=12) 
	else: 
		plt.title(f'Distribution of {feat}', fontsize=12) 
	plt.tight_layout() 
	plt.grid() 


In [ ]:
sns.pairplot(df, hue='label') 

In [ ]:
# sns.heatmap(df.corr(), annot=True)
df_numeric = df.select_dtypes(include=[float, int])

# Correlation matrix
corr_matrix = df_numeric.corr()

# Print the heatmap of matrix
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Matrice de correlation')
plt.show()


## Seperation des features et des target label

In [ ]:
features = df.drop('label', axis=1)
labels = df['label']
features

In [ ]:
# Initialzing empty lists to append all model's name and corresponding name
acc = []
model = []

## Preparation des données d'entrainement et données de test

In [ ]:
# Splitting into train and test data

from sklearn.model_selection import train_test_split

label_encoder = LabelEncoder()
target_encoded = label_encoder.fit_transform(labels)

Xtrain, Xtest, Ytrain, Ytest = train_test_split(features, target_encoded, test_size=0.2, random_state=42)
# Xtrain, Xtest, Ytrain, Ytest = train_test_split(features,target,test_size = 0.2,random_state =2)

# Arbre de décision

In [ ]:
from sklearn.tree import DecisionTreeClassifier

DecisionTree = DecisionTreeClassifier(criterion="entropy",random_state=2,max_depth=5)

DecisionTree.fit(Xtrain,Ytrain)

predicted_values = DecisionTree.predict(Xtest)
x = metrics.accuracy_score(Ytest, predicted_values)
acc.append(x)
model.append('Arbre de Décision')
print("DecisionTrees's Accuracy is: ", x)

print(classification_report(Ytest,predicted_values))

In [ ]:
from sklearn.model_selection import cross_val_score

In [ ]:
# Cross validation score (Decision Tree)
score = cross_val_score(DecisionTree, features, labels,cv=5)

In [ ]:
score

### Sauvegarde du model

In [ ]:
import pickle
# Dump the trained Naive Bayes classifier with Pickle
DT_pkl_filename = '../api/models/DecisionTree.pkl'
# Open the file to save as pkl file
DT_Model_pkl = open(DT_pkl_filename, 'wb')
pickle.dump(DecisionTree, DT_Model_pkl)
# Close the pickle instances
DT_Model_pkl.close()

# Naives Bayes Guassien

In [ ]:
from sklearn.naive_bayes import GaussianNB

NaiveBayes = GaussianNB()

NaiveBayes.fit(Xtrain,Ytrain)

predicted_values = NaiveBayes.predict(Xtest)
x = metrics.accuracy_score(Ytest, predicted_values)
acc.append(x)
model.append('Naive Bayes')
print("Naive Bayes's Accuracy is: ", x)

print(classification_report(Ytest,predicted_values))

In [ ]:
# Cross validation score (NaiveBayes)
score = cross_val_score(NaiveBayes,features,labels,cv=5)
score

### Sauvegarde du model

In [ ]:
import pickle
# Dump the trained Naive Bayes classifier with Pickle
NB_pkl_filename = '../api/models/NBClassifier.pkl'
# Open the file to save as pkl file
NB_Model_pkl = open(NB_pkl_filename, 'wb')
pickle.dump(NaiveBayes, NB_Model_pkl)
# Close the pickle instances
NB_Model_pkl.close()

# Machine à Vecteurs de Support (SVM)

In [ ]:
from sklearn.svm import SVC

SVM = SVC(gamma='auto')

SVM.fit(Xtrain,Ytrain)

predicted_values = SVM.predict(Xtest)

x = metrics.accuracy_score(Ytest, predicted_values)
acc.append(x)
model.append('SVM')
print("SVM's Accuracy is: ", x)

print(classification_report(Ytest,predicted_values))

In [ ]:
# Cross validation score (SVM)
score = cross_val_score(SVM,features,labels,cv=5)
score

# Regression Logistique

In [ ]:
from sklearn.linear_model import LogisticRegression

LogReg = LogisticRegression(random_state=2)

LogReg.fit(Xtrain,Ytrain)

predicted_values = LogReg.predict(Xtest)

x = metrics.accuracy_score(Ytest, predicted_values)
acc.append(x)
model.append('Regression Logistique')
print("Logistic Regression's Accuracy is: ", x)

print(classification_report(Ytest,predicted_values))

In [ ]:
# Cross validation score (Logistic Regression)
score = cross_val_score(LogReg,features,labels,cv=5)
score

### Sauvegarde du model etrainé

In [ ]:
import pickle
# Dump the trained Naive Bayes classifier with Pickle
LR_pkl_filename = '../api/models/LogisticRegression.pkl'
# Open the file to save as pkl file
LR_Model_pkl = open(DT_pkl_filename, 'wb')
pickle.dump(LogReg, LR_Model_pkl)
# Close the pickle instances
LR_Model_pkl.close()

# Foret Aléatoire (RF)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

RF = RandomForestClassifier(n_estimators=20, random_state=0)
RF.fit(Xtrain,Ytrain)

predicted_values = RF.predict(Xtest)

x = metrics.accuracy_score(Ytest, predicted_values)
acc.append(x)
model.append('RF')
print("RF's Accuracy is: ", x)

print(classification_report(Ytest,predicted_values))

In [ ]:
# Cross validation score (Random Forest)
score = cross_val_score(RF,features,labels,cv=5)
score

### Sauvegarde du model entrainé

In [ ]:
import pickle
# Dump the trained Naive Bayes classifier with Pickle
RF_pkl_filename = '../api/models/RandomForest.pkl'
# Open the file to save as pkl file
RF_Model_pkl = open(RF_pkl_filename, 'wb')
pickle.dump(RF, RF_Model_pkl)
# Close the pickle instances
RF_Model_pkl.close()

# XGBoost

In [ ]:
!pip install xgboost

In [ ]:
import xgboost as xgb

XB = xgb.XGBClassifier()
XB.fit(Xtrain, Ytrain)

predicted_values = XB.predict(Xtest)

x = metrics.accuracy_score(Ytest, predicted_values)
acc.append(x)
model.append('XGBoost')
print("XGBoost's Accuracy is: ", x)

print(classification_report(Ytest,predicted_values))

In [ ]:
# Cross validation score (XGBoost)
score = cross_val_score(XB,features,target_encoded,cv=5)
score

### Sauvegarde du model entrainé

In [ ]:
import pickle
# Dump the trained Naive Bayes classifier with Pickle
XB_pkl_filename = '../api/models/XGBoost.pkl'
# Open the file to save as pkl file
XB_Model_pkl = open(XB_pkl_filename, 'wb')
pickle.dump(XB, XB_Model_pkl)
# Close the pickle instances
XB_Model_pkl.close()

## Comparaison des Accuracy

In [ ]:
plt.figure(figsize=[10,5],dpi = 100)
plt.title('Comparaison des Accuracy')
plt.xlabel('Accuracy')
plt.ylabel('Algorithmes')
sns.barplot(x = acc,y = model,palette='dark')

In [ ]:
accuracy_models = dict(zip(model, acc))
for k, v in accuracy_models.items():
    print (k, '-->', v)

## Quelques exemples de prédiction

In [ ]:
data = np.array([[104,18, 30, 23.603016, 60.3, 6.7, 140.91]])
# predictionRF = RF.predict(data)
# print("Random Forest: ", predictionRF)

predictionNB = NaiveBayes.predict(data)
print("Resultat avec les Naives Bayes: ", predictionNB)

In [ ]:
data = np.array([[83, 45, 60, 28, 70.3, 7.0, 150.9]])
# predictionRF = RF.predict(data)
# print("Random Forest: ", predictionRF)

predictionNB = NaiveBayes.predict(data)
print("Resultat avec les Naives Bayes: ", predictionNB)

## Différents labels avec leurs valeurs valeurs encodées

In [ ]:
for label, encoded in zip(label_encoder.classes_, range(len(label_encoder.classes_))):
    print(f'Label: {label}, Encoded: {encoded}')